In [ ]:
!pip install openai rdflib spacy pyvis datasets scikit-learn matplotlib tqdm pandas

In [ ]:
# Import necessary libraries
import os
import re
import json
from collections import Counter
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pandas as pd
import time
import hashlib

# NLP and KG libraries
import spacy
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, XSD, SKOS  # Use SKOS for altLabel

# OpenAI client for LLM
from openai import OpenAI

# Visualization
from pyvis.network import Network

# Hugging Face datasets library
from datasets import load_dataset

# For embedding similarity
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
# Using a specific dataset version helps keep results consistent
cnn_dm_dataset = load_dataset("cnn_dailymail", "3.0.0")


In [ ]:
# Calculate the total number of records
total_records = (
    len(cnn_dm_dataset["train"])
    + len(cnn_dm_dataset["validation"])
    + len(cnn_dm_dataset["test"])
)
print(f"Total number of records in the dataset: {total_records}\n")
print("Sample record from the training dataset:")
print(cnn_dm_dataset["train"][0])

# #### OUTPUT ####
# Total number of records in the dataset: 311971
#
# Sample record from the training dataset:
# {'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel ...'}


In [ ]:
# Define keywords related to tech company acquisitions
ACQUISITION_KEYWORDS = [
    "acquire", "acquisition", "merger", "buyout",
    "purchased by", "acquired by", "takeover"
]
TECH_KEYWORDS = [
    "technology", "software", "startup", "app",
    "platform", "digital", "AI", "cloud"
]


In [ ]:
# Use only the training split
cnn_dm_dataset_train = cnn_dm_dataset["train"]

# Store filtered articles here
filtered_articles = []

# Filter articles by acquisition keywords
for record in cnn_dm_dataset_train:
    found_keyword = False
    for keyword in ACQUISITION_KEYWORDS:
        if keyword.lower() in record["article"].lower():
            found_keyword = True
            break

    if found_keyword:
        filtered_articles.append(record)


In [ ]:
# Print the total number of filtered articles
print(f"Total number of filtered articles: {len(filtered_articles)}")

# Print a sample filtered article
print("\nSample of a filtered article:")
print(filtered_articles[0]["article"])

### OUTPUT ####
# Sample of a filtered article:
# SAN DIEGO, California (CNN) -- You must know whats really driving the
# immigration debate ...


In [ ]:
cleaned_articles = []

for record in filtered_articles:
    text = record["article"]

    # Basic cleanup with regex
    text = re.sub(r"^\(CNN\)\s*(--)?\s*", "", text)
    text = re.sub(r"By .*? for Dailymail\.com.*?Updated:.*", "", text, flags=re.I | re.S)
    text = re.sub(r"PUBLISHED:.*?UPDATED:.*", "", text, flags=re.I | re.S)
    text = re.sub(r"Last updated at.*on.*", "", text, flags=re.I)
    text = re.sub(r"https?://\S+|www\.\S+", "[URL]", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\b[\w.-]+@[\w.-]+\.\w+\b", "[EMAIL]", text)
    text = re.sub(r"\s+", " ", text).strip()

    cleaned_articles.append({
        "id": record["id"],
        "cleaned_text": text,
        "summary": record.get("highlights", "")
    })


### Step 1

In [ ]:
# Download and load the spaCy English model
# Only needs to be run once
spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

# Count entity labels such as PERSON, ORG, and DATE
entity_counts = Counter()

# Run spaCy NER over each cleaned article
for article in cleaned_articles:
    text = article["cleaned_text"]
    doc = nlp(text)

    for ent in doc.ents:
        entity_counts[ent.label_] += 1


In [ ]:
# Extract labels and counts
labels, counts = zip(*entity_counts.items())

# Plot a bar chart
plt.figure(figsize=(12, 7))
plt.bar(labels, counts, color="skyblue")
plt.title("Top Entity Type Distribution (via spaCy)")
plt.ylabel("Frequency")
plt.xlabel("Entity Label")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Step 2

In [ ]:
# Initialize the OpenAI client with your provider configuration
client = OpenAI(
    base_url="YOUR_LLM_API_PROVIDER_URL",
    api_key="YOUR_LLM_API_KEY"
)


In [ ]:
def call_llm(system_prompt, user_prompt, model_name):
    """
    Send a request to the language model (LLM) using the provided prompts.

    Args:
        system_prompt (str): Instructions or context for the LLM.
        user_prompt (str): The user input text to process.
        model_name (str): The model identifier to use.

    Returns:
        str: The text content returned by the model.
    """
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    return response.choices[0].message.content.strip()


In [ ]:
# Keep the top N entity types by frequency
TOP_N_ENTITY_TYPES = 8
relevant_entity_labels_for_llm = [
    label for label, count in entity_counts.most_common(TOP_N_ENTITY_TYPES)
]
entity_types_string_for_prompt = ", ".join(relevant_entity_labels_for_llm)

# System prompt for LLM-based entity extraction
llm_ner_system_prompt = (
    f"You are an expert Named Entity Recognition system. "
    f"From the provided news article text, identify and extract entities. "
    f"The entity types to focus on are: {entity_types_string_for_prompt}. "
    f"For each identified entity, provide its exact text span from the article and its type (use one of the provided types). "
    f"Output only a valid JSON object with a single key named entities. "
    f"The value of entities must be a list of objects with text and type fields. "
    f"If no matching entities are found, return an empty list in the entities field."
)


In [ ]:
def parse_llm_entity_json_output(llm_output_str):
    """
    Parse the LLM JSON string and return the list of extracted entities.

    Expected format:
        {"entities": [{"text": "...", "type": "..."}]}
    """
    if not llm_output_str:
        return []

    if llm_output_str.startswith("```json"):
        llm_output_str = llm_output_str[7:].rstrip("```").strip()

    try:
        data = json.loads(llm_output_str)
        return data.get("entities", [])
    except json.JSONDecodeError:
        return []


In [ ]:
# Define the LLM used for entity extraction
TEXT_GEN_MODEL_NAME = "microsoft/phi-4"
articles_with_llm_entities = []

# Iterate over cleaned articles and extract entities with the LLM
for i, article_data in enumerate(cleaned_articles):
    article_id = article_data["id"]
    article_text = article_data["cleaned_text"]

    llm_response_content = call_llm(
        llm_ner_system_prompt,
        article_text,
        TEXT_GEN_MODEL_NAME
    )

    extracted_llm_entities = []
    if llm_response_content:
        extracted_llm_entities = parse_llm_entity_json_output(llm_response_content)

    articles_with_llm_entities.append({
        "id": article_id,
        "cleaned_text": article_text,
        "summary": article_data["summary"],
        "llm_extracted_entities": extracted_llm_entities
    })


In [ ]:
print(articles_with_llm_entities[0]["llm_extracted_entities"])

# ### OUTPUT ###
# Extracted entities for one article:
# [
#   {
#     "text": "United Nations",
#     "type": "ORG"
#   },
#   {
#     "text": "Algiers",
#     "type": "GPE"
#   }
# ]


### Step 3

In [ ]:
# System prompt for relationship extraction
llm_re_system_prompt = (
    "You are an expert system for extracting relationships between entities from text, "
    "specifically focusing on technology company acquisitions. "
    "Given an article text and a list of pre-extracted named entities (each with 'text' and 'type'), "
    "your task is to identify and extract relationships. "
    "The 'subject_text' and 'object_text' in your output MUST be exact text spans of entities found in the provided 'Extracted Entities' list. "
    "The 'subject_type' and 'object_type' MUST correspond to the types of those entities from the provided list. "
    "Output ONLY a valid JSON object with a single key 'relationships'. The value of 'relationships' MUST be a list of JSON objects. "
    "Each relationship object must have these keys: 'subject_text', 'subject_type', 'predicate', 'object_text', 'object_type'. "
    "Example: {\"relationships\": [{\"subject_text\": \"Innovatech Ltd.\", \"subject_type\": \"ORG\", \"predicate\": \"ACQUIRED\", \"object_text\": \"Global Solutions Inc.\", \"object_type\": \"ORG\"}]} "
    "If no relevant relationships are found between the provided entities, the 'relationships' list should be empty: {\"relationships\": []}."
)


In [ ]:

# {
#   "relationships": [
#     {
#       "subject_text": "Innovatech Ltd.",
#       "subject_type": "ORG",
#       "predicate": "ACQUIRED",
#       "object_text": "Global Solutions Inc.",
#       "object_type": "ORG"
#     }
#   ]
# }

In [ ]:
def parse_llm_relationship_json_output(llm_output_str_rels):
    """
    Parse the LLM JSON string used for relationship extraction.

    Expected format:
        {"relationships": [{"subject_text": ..., "predicate": ..., "object_text": ...}]}
    """
    if not llm_output_str_rels:
        return []

    if llm_output_str_rels.startswith("```json"):
        llm_output_str_rels = llm_output_str_rels[7:].rstrip("```").strip()

    try:
        data = json.loads(llm_output_str_rels)
        return data.get("relationships", [])
    except json.JSONDecodeError:
        return []


In [ ]:
# Reuse the same helper for relationship extraction
call_llm_for_relationships = call_llm
articles_with_llm_relations = []

# Iterate through the articles and extract relationships
for i, article_entity_data in enumerate(articles_with_llm_entities):
    article_id_rels = article_entity_data["id"]
    article_text_rels = article_entity_data["cleaned_text"]
    current_entities = article_entity_data["llm_extracted_entities"]

    entities_json_for_prompt = json.dumps(current_entities)

    user_prompt_for_re = (
        f"Article Text:\n```\n{article_text_rels}\n```\n\n"
        f"Extracted Entities (use these exact texts for subjects/objects of relationships):\n```json\n{entities_json_for_prompt}\n```\n\n"
        "Identify and extract relationships between these entities based on the system instructions."
    )

    llm_response_rels_content = call_llm_for_relationships(
        llm_re_system_prompt,
        user_prompt_for_re,
        TEXT_GEN_MODEL_NAME
    )

    extracted_llm_rels = []
    if llm_response_rels_content:
        extracted_llm_rels = parse_llm_relationship_json_output(llm_response_rels_content)

    articles_with_llm_relations.append({
        **article_entity_data,
        "llm_extracted_relationships": extracted_llm_rels
    })

# Review one sample article's extracted relationships
print(articles_with_llm_relations[0]["llm_extracted_relationships"])

# ### OUTPUT ###
# Extracted LLM relationships for one article:
# [
#   {
#     "subject_text": "Microsoft Corp.",
#     "subject_type": "ORG",
#     "predicate": "ACQUIRED",
#     "object_text": "Nuance Communications Inc.",
#     "object_type": "ORG"
#   },
#   {
#     "subject_text": "Nuance Communications Inc.",
#     "subject_type": "ORG",
#     "predicate": "HAS_PRICE",
#     "object_text": "$19.7 billion",
#     "object_type": "MONEY"
#   }
# ]
# At this point, we have extracted both entities (nodes) and relationships (edges)
# from the article collection, which gives us the core ingredients for a knowledge graph.


In [ ]:
#!/usr/bin/env python3
"""
Bulk-import keywords into RediSearch autocomplete, then query them.
"""

from pathlib import Path
from redis import Redis
from redis.commands.search.suggestion import Suggestion

##############################################################################
# 1 ▸ connection
##############################################################################

r = Redis(host="localhost", port=6379, decode_responses=True)
ac  = r.ft()                     # we only need the helper object, no schema

##############################################################################
# 2 ▸ importer
##############################################################################

def add_keywords_from_file(
    filepath: str | Path,
    key: str,
    batch: int = 10_000,
) -> None:
    """
    Read KEYWORD <TAB|SPACE|COMMA> COUNT from *filepath* and insert into
    the RediSearch autocomplete dictionary stored under *key*.
    """
    filepath = Path(filepath)

    with filepath.open(encoding="utf-8") as fh:
        pipe   = r.pipeline(transaction=False)
        total  = 0

        for i, line in enumerate(fh, 1):
            # -------------------- parse "keyword  count"
            parts = [p.strip() for p in line.strip().replace(",", " ").split()]
            if len(parts) < 2 or not parts[-1].isdigit():
                continue                          # skip malformed lines
            *token_parts, count_str = parts
            token  = " ".join(token_parts)
            score  = float(count_str)             # RediSearch wants a float

            # -------------------- stage FT.SUGADD
            pipe.ft().sugadd(
                key,
                Suggestion(token, score=score),
                # INCR=True makes repeated terms accumulate counts
                increment=True,
            )
            total += 1

            if i % batch == 0:
                pipe.execute()
                pipe = r.pipeline(transaction=False)

        pipe.execute()

    print(f"Inserted/updated {total:,} keywords into '{key}'")


##############################################################################
# 3 ▸ query helper
##############################################################################

def suggest(
    key: str,
    prefix: str,
    *,
    max_results: int = 10,
    fuzzy: bool = True,
):
    """
    Return up to *max_results* suggestions for *prefix*.
    """
    results = ac.sugget(
        key,
        prefix,
        max=max_results,
        fuzzy=fuzzy,
        with_scores=True,
        with_payloads=False,
    )
    # results is a list of Suggestion objects
    return [(s.string, s.score) for s in results]


##############################################################################
# 4 ▸ demo
##############################################################################

if __name__ == "__main__":
    DICT_KEY = "keywords_ac"

    add_keywords_from_file("keywords_counts.txt", DICT_KEY)

    for p in ("stra", "strawb", "jam"):
        print(f"\nSuggestions for '{p}':")
        for term, score in suggest(DICT_KEY, p):
            print(f"  {term:<30s} {score:,.0f}")



import nltk
from nltk.data import find

# List all NLTK resources you may need across your projects
NLTK_RESOURCES = [
    "corpora/stopwords",
    "corpora/wordnet",
    "tokenizers/punkt",
    "taggers/averaged_perceptron_tagger",
    # Add more if needed
]

def ensure_nltk_resources():
    """Download required NLTK resources safely (only if missing)."""
    for resource in NLTK_RESOURCES:
        try:
            find(resource)
        except LookupError:
            nltk.download(resource.split("/", 1)[1], quiet=True)

# Call once during module load or app startup
ensure_nltk_resources()